# 🏥 AMOS22 L3 VFA/PMA Complete Pipeline

## İki Aşamalı Pipeline:
1. **Kural Tabanlı**: L3 tespit + Fasya + HU banding + VFA/PMA hesaplama
2. **Derin Öğrenme**: 60 epoch U-Net eğitimi (teacher: kural tabanlı)

**Veri**: AMOS221 CT + TS abdominal_muscles segmentleri

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/L3_SO_ANALYSIS')
print(f"📂 Çalışma dizini: {os.getcwd()}")

In [ ]:
!pip install SimpleITK nibabel scikit-image opencv-python-headless scipy tqdm pandas torch torchvision monai -q

import numpy as np
import SimpleITK as sitk
import cv2
from scipy import ndimage
from skimage import morphology, measure, segmentation
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import json
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from monai.losses import DiceLoss
from monai.networks.nets import UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Kütüphaneler yüklendi")
print(f"🖥️ Device: {device}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")

---
# BÖLÜM 1: KURAL TABANLI PIPELINE
---

In [ ]:
AMOS_ROOT = Path("/content/drive/MyDrive/AMOS221/imagesTr")
TS_ROOT = Path("/content/drive/MyDrive/TS_teachers_AMOS22")
OUTPUT_ROOT = Path("/content/drive/MyDrive/L3_SO_ANALYSIS/amos_results")
QA_DIR = OUTPUT_ROOT / "qa_overlays"
MODEL_DIR = OUTPUT_ROOT / "dl_models"

OUTPUT_ROOT.mkdir(exist_ok=True, parents=True)
QA_DIR.mkdir(exist_ok=True, parents=True)
MODEL_DIR.mkdir(exist_ok=True, parents=True)

amos_cases = sorted(AMOS_ROOT.glob("amos_*.nii.gz"))
print(f"�� AMOS vaka sayısı: {len(amos_cases)}")
print(f"📂 Çıktı dizini: {OUTPUT_ROOT}")